# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# --- Signal check 1 (flag-linked): staleness behind FlyRank's refresh flag ---
# Claim: pages that haven't been updated in a while are more likely to be declining.
# This is the assumption baked into FlyRank's hand-written refresh flag.
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

signal1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'size'),
    decline_rate=('is_declining', 'mean'),
).round(3)
print("Signal 1 -- freshness_tier vs decline rate (flag-linked: staleness)")
print(signal1)
print(
    "\nVerdict: CONFIRMED (on the two well-populated buckets) -- pages last "
    "updated 91-180 days ago decline at 61.1% (n=9,171) vs 51.1% for pages "
    "updated in the last 30 days (n=20,480). The 31-90 and 181+ buckets have "
    "~175 rows each -- too small to trust on their own, so the verdict rests "
    "on the two buckets with real sample size."
)

# --- Signal check 2 (my own): word count, the idea behind my Week-1 lane pitch ---
# Claim I made in ML-02: "long content gets low traffic, so length itself is the problem."
# Testing that claim directly instead of assuming it still holds.
signal2 = df.groupby('word_count_tier').agg(
    n=('content_id', 'size'),
    avg_sessions=('sessions_90d', 'mean'),
    median_sessions=('sessions_90d', 'median'),
).round(1)
print("\nSignal 2 -- word_count_tier vs sessions_90d (my Week-1 claim: length hurts traffic)")
print(signal2)
print(
    "\nVerdict: OPPOSITE -- longer pages get MORE sessions, not fewer. "
    "3500+-word pages average 82.4 sessions (median 22.0, n=6,285) vs 2.1 "
    "average (median 1.0, n=973) for pages under 1,000 words. My Week-1 claim "
    "compared two overall averages side by side, not the actual per-page "
    "relationship -- word count does not belong in the rule. A clean negative "
    "here saves the rule from leaning on a signal that doesn't hold."
)

Signal 1 -- freshness_tier vs decline rate (flag-linked: staleness)
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
181+              174         0.471
31-90             175         0.589
91-180           9171         0.611

Verdict: CONFIRMED (on the two well-populated buckets) -- pages last updated 91-180 days ago decline at 61.1% (n=9,171) vs 51.1% for pages updated in the last 30 days (n=20,480). The 31-90 and 181+ buckets have ~175 rows each -- too small to trust on their own, so the verdict rests on the two buckets with real sample size.

Signal 2 -- word_count_tier vs sessions_90d (my Week-1 claim: length hurts traffic)
                     n  avg_sessions  median_sessions
word_count_tier                                      
1000-2000         3780          12.4              4.0
2000-3500        11263          28.7              6.0
3500+             6285          82.4             22.0
<1000              973           2.

**My rule, in plain words:** a page is worth a refresh review if it hasn't been
touched in a while *and* it still gets real search visibility. Signal 1 shows that staler
pages decline more often (backing the "stale = risk" half), and visibility (impressions)
tells us the page is worth the effort. Signal 2 (word count) turned out **OPPOSITE**, so
length is left out of the rule entirely — a page's length doesn't predict whether it needs
attention here.

**Reason code:** every row scored by this rule gets the same code, `stale_and_visible_refresh_candidate`
— it is ONE rule, so it carries one reason, not a menu of them. The score itself (Section 2)
is what separates a borderline case from an obvious one.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os

# Score: how overdue (freshness_risk) x how visible (visibility), each as a 0-1 percentile
# rank across the whole dataset. Multiplying (not averaging) means a page has to score on
# BOTH dimensions to rank high -- a very stale but invisible page, or a very visible but
# freshly-updated page, won't outrank one that is genuinely both.
freshness_risk = df['days_since_last_update'].rank(method='average', pct=True)
visibility = np.log1p(df['impressions_90d']).rank(method='average', pct=True)

df['freshness_risk_pctl'] = freshness_risk.round(4)
df['visibility_pctl'] = visibility.round(4)
df['baseline_action_score'] = (freshness_risk * visibility).round(4)
df['reason_code'] = 'stale_and_visible_refresh_candidate'

# Action label: three tiers off the same single score -- still one rule, just thresholds on it.
q75 = df['baseline_action_score'].quantile(0.75)
q50 = df['baseline_action_score'].quantile(0.50)

def action_label(score):
    if score >= q75:
        return 'refresh_now'
    elif score >= q50:
        return 'monitor'
    return 'no_action_needed'

df['action_label'] = df['baseline_action_score'].apply(action_label)
df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)

queue_cols = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
    'freshness_risk_pctl', 'visibility_pctl', 'reason_code', 'action_label',
    'days_since_last_update', 'freshness_tier', 'impressions_90d', 'impression_tier',
    'sessions_90d', 'avg_position', 'content_type', 'main_intent', 'word_count',
]
queue = df.sort_values('baseline_rank')[queue_cols]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
print("\naction_label distribution:")
print(df['action_label'].value_counts())
print(f"\nq50={q50:.4f}  q75={q75:.4f}")

Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv

action_label distribution:
action_label
no_action_needed    14999
monitor              7501
refresh_now          7500
Name: count, dtype: int64

q50=0.2127  q75=0.4128


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top10 = queue.head(10)
print(top10[[
    'baseline_rank', 'baseline_action_score', 'action_label',
    'days_since_last_update', 'impressions_90d', 'sessions_90d',
    'avg_position', 'content_type',
]].to_string(index=False))

 baseline_rank  baseline_action_score action_label  days_since_last_update  impressions_90d  sessions_90d  avg_position    content_type
             1                 0.9827  refresh_now                     194            61678           119          19.7 keyword article
             2                 0.9825  refresh_now                     106            79035            44           8.7 keyword article
             3                 0.9823  refresh_now                     194            59472            82          24.8 keyword article
             4                 0.9681  refresh_now                     106            40305           408          28.4 keyword article
             5                 0.9518  refresh_now                     194            25715            80          22.2 keyword article
             6                 0.9517  refresh_now                     106            28000           345           4.7 keyword article
             7                 0.9484  refresh_n

**Top-10 review** — action, why it's there, what would make it wrong:

1. **refresh_now** — 194 days stale, 61.7k impressions (most visible page in the set). Wrong if: the page is already scheduled for a redesign that would replace this refresh anyway.
2. **refresh_now** — 106 days stale, 79k impressions, page-1 position (8.7). Wrong if: `trend_direction` here is `stable`, not `down` — the rule doesn't check trend, so this may be a page that isn't actually declining and gets over-prioritized.
3. **refresh_now** — 194 days stale, 59.5k impressions, position 24.8 (page 3-5). Wrong if: position 24.8 with only moderate CTR means the real fix is a keyword/targeting problem, not a content refresh.
4. **refresh_now** — 106 days stale, 40k impressions, decent CTR (0.96%). Wrong if: the page is already performing fine on CTR and the "problem" is really about lower search demand than assumed.
5. **refresh_now** — 194 days stale, 25.7k impressions, position 22.2. Wrong if: `main_intent` is informational content already well-served, so a refresh brings little upside versus a page that ranks for commercial intent.
6. **refresh_now** — 106 days stale, position 4.7 (already page 1). Wrong if: engagement_rate is very low (0.58%) — the page draws clicks but people bounce; a content refresh may not be the actual fix (could be UX/landing issue).
7. **refresh_now** — 106 days stale, position 13.2, CTR 2.43%. Wrong if: the page's high CTR relative to position suggests it's already working reasonably well and doesn't need urgent attention.
8. **refresh_now** — 106 days stale, position 4.6, but engagement_rate 0.0%. Wrong if: 0% engagement could mean an instrumentation gap rather than a real content problem — worth checking before treating it as a refresh candidate.
9. **refresh_now** — 151 days stale, but `trend_direction` is `up`, not `down`. Wrong if: this page is actively improving — refreshing it now could be premature effort better spent on a declining page.
10. **refresh_now** — 106 days stale, position 31.7 (page 3-5). Wrong if: position 31.7 is far enough down that visibility gains from a refresh alone (without a ranking/targeting fix) may be limited.

Rows 2 and 9 are flagged explicitly as weak picks in Section 4 — they pass the rule's staleness+visibility test but don't show the decline this rule assumes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
print("Weak picks (rank 2 and rank 9) -- trend_direction, for context only, never a feature:")
print(queue.iloc[[1, 8]][['baseline_rank', 'content_id']].merge(
    df[['content_id', 'trend_direction', 'avg_position']], on='content_id'
).to_string(index=False))

print("\nLeakage check -- inputs used in baseline_action_score:")
print("  days_since_last_update  -- static metadata, known before any 90-day window closes")
print("  impressions_90d         -- trailing 90-day total, not a future window")
print("\nInputs explicitly NOT used as features:")
print("  trend_direction / trend_pct  -- these DEFINE is_declining_label; used only to")
print("                                  audit Signal 1 above, never as rule inputs")
print("  health_score / priority_score / action_type / refresh flags -- FlyRank's own")
print("                                  product outputs; not present in this dataset,")
print("                                  and would be circular if they were")

Weak picks (rank 2 and rank 9) -- trend_direction, for context only, never a feature:
 baseline_rank           content_id trend_direction  avg_position
             2 content_a5dbb404bdc2          stable           8.7
             9 content_cb7e312f5d32              up          12.6

Leakage check -- inputs used in baseline_action_score:
  days_since_last_update  -- static metadata, known before any 90-day window closes
  impressions_90d         -- trailing 90-day total, not a future window

Inputs explicitly NOT used as features:
  trend_direction / trend_pct  -- these DEFINE is_declining_label; used only to
                                  audit Signal 1 above, never as rule inputs
  health_score / priority_score / action_type / refresh flags -- FlyRank's own
                                  product outputs; not present in this dataset,
                                  and would be circular if they were


**Weak picks:** rank 2 (`trend_direction = stable`) and rank 9 (`trend_direction = up`)
both clear the stale-and-visible bar but aren't actually declining — the rule has no way
to see that, since staleness and visibility say nothing about direction. That's the honest
cost of a two-input rule: it will occasionally flag a healthy page.

**Leakage check:** the score uses only `days_since_last_update` and `impressions_90d` —
both knowable without looking at `trend_direction`/`trend_pct` (the label source) or any
future window. FlyRank's own product flags (`health_score`, `priority_score`, `action_type`)
aren't in this dataset and were never used as inputs, only as the framing this rule is
meant to be compared against.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.